# WealthBot — DistilBERT Transaction Categorizer (Colab)

Fine-tunes **`distilbert-base-uncased`** from HuggingFace to classify transaction text
(`merchant_name + description`) into **15 spending categories** for Indian students.

**Strategy:** Freeze all DistilBERT base layers → train only the classification head (0.9% of params).

**Runtime:** ~3 min on T4 GPU.

**Outputs:** `categorizer.onnx`, `tokenizer/`, `label_encoder.json` — download and place in `ml/models/`.

---
⚠️ **Make sure to set Runtime → T4 GPU** before running.

In [ ]:
!pip install transformers torch onnx onnxruntime onnxscript pandas numpy -q

In [ ]:
# Upload synthetic_transactions.csv from ml/models/
from google.colab import files
uploaded = files.upload()  # Select: synthetic_transactions.csv

In [ ]:
import json
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizerFast,
    get_linear_schedule_with_warmup,
)

# === Paths (Colab) ===
CSV_PATH = Path("synthetic_transactions.csv")
ONNX_OUTPUT_PATH = Path("categorizer.onnx")
TOKENIZER_OUTPUT_DIR = Path("tokenizer")
LABEL_ENCODER_PATH = Path("label_encoder.json")

# === Hyperparameters ===
MAX_LENGTH = 64
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 5e-4
WARMUP_RATIO = 0.1
BASE_MODEL = "distilbert-base-uncased"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# === Data Preparation ===

df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df):,} transactions")

df["text"] = (
    df["merchant_name"].fillna("").astype(str)
    + " "
    + df["description"].fillna("").astype(str)
)

all_categories = sorted(df["category"].unique().tolist())
label2id = {cat: idx for idx, cat in enumerate(all_categories)}
id2label = {idx: cat for cat, idx in label2id.items()}
n_labels = len(all_categories)
print(f"Categories ({n_labels}): {all_categories}")

df["label"] = df["category"].map(label2id)

# 80/20 split by user (no data leakage between users)
user_ids = df["user_id"].unique()
rng = np.random.default_rng(42)
rng.shuffle(user_ids)
split_idx = int(len(user_ids) * 0.8)
train_users = set(user_ids[:split_idx])

train_df = df[df["user_id"].isin(train_users)].reset_index(drop=True)
val_df = df[~df["user_id"].isin(train_users)].reset_index(drop=True)

train_texts, train_labels = train_df["text"].tolist(), train_df["label"].tolist()
val_texts, val_labels = val_df["text"].tolist(), val_df["label"].tolist()

print(f"Train: {len(train_texts):,}  |  Val: {len(val_texts):,}")

In [ ]:
# === Dataset & DataLoaders ===

class TransactionDataset(Dataset):
    """PyTorch dataset that tokenizes text on-the-fly."""

    def __init__(self, texts, labels, tokenizer, max_length=MAX_LENGTH):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }


tokenizer = DistilBertTokenizerFast.from_pretrained(BASE_MODEL)

train_ds = TransactionDataset(train_texts, train_labels, tokenizer)
val_ds = TransactionDataset(val_texts, val_labels, tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")

In [ ]:
# === Model Setup (freeze base, train head only) ===

model = DistilBertForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=n_labels,
    id2label=id2label,
    label2id=label2id,
)

# Freeze all DistilBERT base layers
for param in model.distilbert.parameters():
    param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total:,} total, {trainable:,} trainable ({100 * trainable / total:.1f}%)")

model = model.to(device)

In [ ]:
# === Training Functions ===

def train_epoch(model, dataloader, optimizer, scheduler, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item() * input_ids.size(0)

        preds = outputs.logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate_epoch(model, dataloader, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        total_loss += outputs.loss.item() * input_ids.size(0)
        preds = outputs.logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total

In [ ]:
# === Training Loop ===

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=0.01,
)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

best_val_acc = 0.0
best_state = {}

print(f"Training for {EPOCHS} epochs (batch={BATCH_SIZE}, lr={LEARNING_RATE}, warmup={warmup_steps})")
print("-" * 75)

start = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss, val_acc = evaluate_epoch(model, val_loader, device)

    marker = ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        marker = " \u2605"

    print(
        f"  Epoch {epoch:2d}/{EPOCHS}  "
        f"train_loss={train_loss:.4f}  train_acc={train_acc:.4f}  |  "
        f"val_loss={val_loss:.4f}  val_acc={val_acc:.4f}{marker}"
    )

elapsed = time.perf_counter() - start
print(f"\nTraining completed in {elapsed:.1f}s (best val accuracy: {best_val_acc:.4f})")

# Restore best weights
if best_state:
    model.load_state_dict(best_state)
    print("Restored best checkpoint.")

In [ ]:
# === ONNX Export (single-file) ===

import onnx
import onnxruntime as ort

model.eval()
model = model.to("cpu")

dummy_input = tokenizer(
    "Swiggy Food purchase",
    max_length=MAX_LENGTH,
    padding="max_length",
    truncation=True,
    return_tensors="pt",
)
input_ids = dummy_input["input_ids"]
attention_mask = dummy_input["attention_mask"]

# Export (may produce external .data file for large models)
tmp_path = "categorizer_tmp.onnx"
print(f"Exporting to ONNX \u2192 {ONNX_OUTPUT_PATH}")

torch.onnx.export(
    model,
    (input_ids, attention_mask),
    tmp_path,
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch_size"},
        "attention_mask": {0: "batch_size"},
        "logits": {0: "batch_size"},
    },
    opset_version=14,
    do_constant_folding=True,
)

# Consolidate into a single .onnx file (no external .data)
onnx_model = onnx.load(tmp_path, load_external_data=True)
onnx.save_model(
    onnx_model,
    str(ONNX_OUTPUT_PATH),
    save_as_external_data=False,
)
# Clean up temp files
import glob
for f in glob.glob("categorizer_tmp*"):
    Path(f).unlink(missing_ok=True)

# Verify ONNX matches PyTorch
session = ort.InferenceSession(str(ONNX_OUTPUT_PATH))
with torch.no_grad():
    pt_logits = model(input_ids, attention_mask).logits.numpy()
onnx_logits = session.run(None, {
    "input_ids": input_ids.numpy(),
    "attention_mask": attention_mask.numpy(),
})[0]

max_diff = float(np.max(np.abs(pt_logits - onnx_logits)))
status = "\u2713" if max_diff < 0.01 else "\u2717 WARN"
print(f"ONNX verification \u2014 max abs diff: {max_diff:.6f} {status}")
print(f"ONNX model saved ({ONNX_OUTPUT_PATH.stat().st_size / 1024 / 1024:.1f} MB)")

In [ ]:
# === Save Tokenizer & Label Encoder ===

TOKENIZER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
tokenizer.save_pretrained(str(TOKENIZER_OUTPUT_DIR))
print(f"Tokenizer saved \u2192 {TOKENIZER_OUTPUT_DIR}/")

encoder = {
    "label2id": label2id,
    "id2label": {str(k): v for k, v in id2label.items()},
    "num_labels": len(label2id),
}
with open(LABEL_ENCODER_PATH, "w") as f:
    json.dump(encoder, f, indent=2, ensure_ascii=False)
print(f"Label encoder saved \u2192 {LABEL_ENCODER_PATH}")

print("\n\u2705 Training pipeline complete!")

## Download Artifacts

Run the cell below to download the 3 artifacts. Place them in your project:

| Artifact | Destination |
|----------|-------------|
| `categorizer.onnx` | `ml/models/categorizer.onnx` |
| `label_encoder.json` | `ml/models/label_encoder.json` |
| `tokenizer.zip` (unzip) | `ml/models/tokenizer/` |

In [ ]:
import shutil

# Zip the tokenizer folder
shutil.make_archive("tokenizer", "zip", ".", "tokenizer")

# Download all 3 artifacts
files.download("categorizer.onnx")
files.download("label_encoder.json")
files.download("tokenizer.zip")

print("Downloaded! Unzip tokenizer.zip into ml/models/tokenizer/")